In [0]:
from pyspark.sql import functions as F

df_variants         = spark.table("silver_clinvar.silver_clinvar_variants")
df_locations        = spark.table("silver_clinvar.silver_clinvar_locations")
df_genes            = spark.table("silver_clinvar.silver_clinvar_genes")
df_traits           = spark.table("silver_clinvar.silver_clinvar_traits")
df_assertions       = spark.table("silver_clinvar.silver_clinvar_assertions")

In [0]:

dim_variants = (df_variants
    .join(df_locations, on="variation_id", how="inner")
    .select(
        "variation_id",
        "variation_name",
        "variation_type",
        "hgvs_name",
        "chromosome",
        "start_pos"
    )
    .dropDuplicates(["variation_id"])
)

dim_genes = (df_genes
    .select("gene_id", "gene_symbol", "gene_name")
    .dropDuplicates(["gene_id"])
    .withColumn("sk_dim_gene", F.sha2(F.col("gene_id").cast("string"), 256))
)

dim_traits = (df_traits
              .filter(F.col("medgen_cui").isNotNull())
              .select(F.sha2(F.col("clinical_assertion_id").cast("string"), 256).alias("sk_dim_traits"), 
                      "medgen_name",
                      "medgen_cui",
                      "trait_type")
)

In [0]:
fato_submissions = (df_assertions.alias("a")
    .join(
        dim_variants.select("variation_id").alias("v"),
        on="variation_id",
        how="inner"
    )
    .select(
        F.sha2(F.col("a.assertion_id").cast("string"), 256).alias("sk_fato_laudo"),
        "a.variation_id",
        "a.assertion_id",
        "a.submitter_name",
        F.col("a.classification_germline_classification").alias("classificacao_laboratorio"),
        F.col("a.classification_review_status").alias("status_revisao_laboratorio"),
        F.col("a.contributes_to_classification").alias("contribui_para_consenso"),
        "a.method_type",
        "a.citation_id",
        "a.date_last_updated",
        "a.ingestion_date"
    )
)

In [0]:
dim_variants.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_clinvar.dim_variants_gold")
dim_genes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_clinvar.dim_genes_gold")
dim_traits.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_clinvar.dim_traits_gold")
fato_submissions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_clinvar.fato_submissoes")

print("Tabelas Ouro modeladas e filtradas com sucesso!")

Tabelas Ouro modeladas e filtradas com sucesso!
